In [16]:
import os
import requests
from time import sleep

from zoneinfo import ZoneInfo
from astral import LocationInfo
from astral.sun import sun
from datetime import datetime, time, date, timedelta

import os
import requests
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from time import sleep
from dateutil import parser
from dotenv import load_dotenv
import json

import numpy as np
from collections import defaultdict, Counter
import math

import drive

In [2]:
lat=latitude = '37.4277'
lon=longitude = '-122.1701'
USER_AGENT = {"User-Agent": "Pikachu"}
tzone= ZoneInfo("America/Los_Angeles")

In [3]:
# A single assumption event will generate 29 prediction entries and 2 real entries; 
assumption_date,assumption_index=drive.time_to_date_index( 
    datetime.now(tzone),lat, lon,tzone )

# Forecast

In [4]:
# Forecast: Probability, wind, humidity , overview 
hourlyForecast = drive.getHourlyForecast(latitude, longitude, USER_AGENT)

In [5]:
drive.time_to_date_index( 
    datetime.fromisoformat(hourlyForecast['properties']['generatedAt'])
    ,lat, lon,tzone
    )

(datetime.date(2026, 1, 5), 3)

In [6]:
groups = defaultdict(list)

for hour in hourlyForecast["properties"]["periods"]:
    dt = datetime.fromisoformat(hour["startTime"])  # aware
    d_local, idx = drive.time_to_date_index(dt, lat, lon, tzone)

    temp_c = (hour["temperature"] - 32) * 5 / 9
    pop = hour.get("probabilityOfPrecipitation", {}).get("value", None)

    # '7 mph' -> 7 -> km/h
    wind_kmh = float(hour["windSpeed"].split()[0]) * 1.60934

    groups[(d_local, idx)].append({
        "startTime": dt.astimezone(tzone),
        "temp_C": temp_c,
        "pop_pct": pop,
        "wind_kmh": wind_kmh,
        "wind_dir": hour.get("windDirection"),
        "forecast": hour.get("shortForecast"),
    })

In [7]:
def mode_or_none(values):
    vals = [v for v in values if v is not None]
    if not vals:
        return None
    return Counter(vals).most_common(1)[0][0]

In [99]:
grouped = []

for (d_local, idx), items in sorted(groups.items(), key=lambda x: (x[0][0], x[0][1])):
    temps = [it["temp_C"] for it in items]
    pops  = [it["pop_pct"] for it in items]
    winds = [it["wind_kmh"] for it in items]
    dirs  = [it["wind_dir"] for it in items]
    fcsts = [it["forecast"] for it in items]

    grouped.append({
        "date": d_local,
        "index": idx,
        
        "start_time": min(it["startTime"] for it in items),
        "end_time":  max(it["startTime"] for it in items)+ timedelta(hours=1),

        "temp_C_mean": np.mean(temps),
        "temp_C_min":  np.min(temps),
        "temp_C_max":  np.max(temps),

        "precipation": np.max(pops),

        "wind_kmh_mean": np.mean(winds),
        "wind_dir_mode": mode_or_none(dirs),
        "forecast_mode": mode_or_none(fcsts),
    })


# Raw Data 

In [19]:
raw = drive.getOriginalData(latitude, longitude, USER_AGENT)

In [95]:
def build_grouped_from_values(values, field_name, *, lat, lon, tzone, drive, agg="mean"):
	groups = defaultdict(list)

	for hour in values:
		dt = datetime.fromisoformat(hour["validTime"].split("/")[0])  # aware
		d_local, idx = drive.time_to_date_index(dt, lat, lon, tzone)

		v = hour.get("value", None)
		groups[(d_local, idx)].append({
			"startTime": dt.astimezone(tzone),
			"value": v
		})

	grouped = []
	for (d_local, idx), items in sorted(groups.items(), key=lambda x: (x[0][0], x[0][1])):
		vals = np.array(
			[np.nan if it["value"] is None else float(it["value"]) for it in items],
			dtype=float
		)

		start_time = min(it["startTime"] for it in items)
		end_time   = max(it["startTime"] for it in items) + timedelta(hours=1)

		all_nan = (vals.size == 0) or np.all(np.isnan(vals))

		if agg == "sum":
			stat = float(np.nansum(vals)) if not all_nan else None
		elif agg == "mean":
			stat = float(np.nanmean(vals)) if not all_nan else None
		else:
			raise ValueError(f"agg must be 'mean' or 'sum', got {agg!r}")

		grouped.append({
			"date": d_local,
			"index": idx,
			"n_hours": len(items),
			"start_time": start_time,
			"end_time": end_time,
			f"{field_name}": stat,
		})

	return groups, grouped

In [96]:
groups_snowLevel, grouped_snowLevel = build_grouped_from_values(
	raw["properties"]["snowLevel"]["values"],
	"snowLevel",
	lat=lat, lon=lon, tzone=tzone, drive=drive, agg="mean"
)

groups_snowfallAmount, grouped_snowfallAmount = build_grouped_from_values(
	raw["properties"]["snowfallAmount"]["values"],
	"snowfallAmount",
	lat=lat, lon=lon, tzone=tzone, drive=drive, agg="sum"
)

groups_quantitativePrecipitation, grouped_quantitativePrecipitation = build_grouped_from_values(
	raw["properties"]["quantitativePrecipitation"]["values"],
	"quantitativePrecipitation",
	lat=lat, lon=lon, tzone=tzone, drive=drive, agg="sum"
)

groups_skyCover, grouped_skyCover = build_grouped_from_values(
	raw["properties"]["skyCover"]["values"],
	"skyCover",
	lat=lat, lon=lon, tzone=tzone, drive=drive, agg="mean"
)

groups_ice, grouped_ice = build_grouped_from_values(
	raw["properties"]["iceAccumulation"]["values"],
	"iceAccumulation",
	lat=lat, lon=lon, tzone=tzone, drive=drive, agg="mean"
)

In [107]:
def build_value_map(grouped_list, key_name: str):
	m = {}
	for row in grouped_list:
		m[(row["date"], row["index"])] = row.get(key_name, None)
	return m

sky_map     = build_value_map(grouped_skyCover, "skyCover")
snowlvl_map = build_value_map(grouped_snowLevel, "snowLevel")
snowamt_map = build_value_map(grouped_snowfallAmount, "snowfallAmount")
qpf_map     = build_value_map(grouped_quantitativePrecipitation, "quantitativePrecipitation")
ice_map     = build_value_map(grouped_ice, "iceAccumulation")


for row in grouped:
	key = (row["date"], row["index"])

	row["skyCover"] = sky_map.get(key, None)
	row["snowLevel"] = snowlvl_map.get(key, None)
	row["snowfallAmount"] = snowamt_map.get(key, None)
	row["quantitativePrecipitation"] = qpf_map.get(key, None)
	row["iceAccumulation"] = ice_map.get(key, None)

In [ ]:
grouped